In [3]:
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from IPython.display import clear_output
from sklearn import datasets
from torch import nn, optim
from torchvision.datasets import MNIST
from typing_extensions import Literal

%matplotlib inline
device = "cuda:0" if torch.cuda.is_available() else "cpu"



In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Head(nn.Module):
    def __init__(self, in_features, out_features, dropout_rate):
        super(Head, self).__init__()
        self.key = nn.Linear(in_features, out_features, bias=False)
        self.query = nn.Linear(in_features, out_features, bias=False)
        self.value = nn.Linear(in_features, out_features, bias=False)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x):
        keys = self.key(x)
        queries = self.query(x)
        values = self.value(x)

        # Scaled dot-product attention
        scores = torch.matmul(queries, keys.transpose(-2, -1)) / (keys.size(-1) ** 0.5)
        attention = F.softmax(scores, dim=-1)
        attention = self.dropout(attention)

        out = torch.matmul(attention, values)
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, in_features, num_heads, dropout_rate):
        super(MultiHeadAttention, self).__init__()
        self.heads = nn.ModuleList([Head(in_features, in_features // num_heads, dropout_rate) for _ in range(num_heads)])
        self.fc_out = nn.Linear(in_features, in_features)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        out = self.fc_out(out)
        return self.dropout(out)

class FeedForward(nn.Module):
    def __init__(self, in_features, hidden_features, dropout_rate):
        super(FeedForward, self).__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.activation = F.gelu
        self.fc2 = nn.Linear(hidden_features, in_features)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x):
        return self.dropout(self.fc2(self.activation(self.fc1(x))))

class TransformerBlock(nn.Module):
    def __init__(self, in_features, num_heads, hidden_features, dropout_rate):
        super(TransformerBlock, self).__init__()
        self.attention = MultiHeadAttention(in_features, num_heads, dropout_rate)
        self.ff = FeedForward(in_features, hidden_features, dropout_rate)
        self.layer_norm1 = nn.LayerNorm(in_features)
        self.layer_norm2 = nn.LayerNorm(in_features)

    def forward(self, x):
        attention_out = self.attention(x)
        x = self.layer_norm1(x + attention_out)  # Residual connection
        forward_out = self.ff(x)
        return self.layer_norm2(x + forward_out)  # Residual connection

# Example usage
if __name__ == "__main__":
    in_features = 384
    num_heads = 6
    hidden_features = 1536
    dropout_rate = 0.2

    model = TransformerBlock(in_features, num_heads, hidden_features, dropout_rate)
    x = torch.rand(10, 20, in_features)  # Batch size: 10, Sequence length: 20, Features: 384
    output = model(x)
    print(output.shape)  # Should print: torch.Size([10, 20, 384])


torch.Size([10, 20, 384])
